# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.5 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 2880
Session ID: 3434a6db-8a5d-4d2e-aad6-51d995e641e9
Applying the following default arguments:
--glue_kernel_version 1.0.5
--enable-glue-datacatalog true
Waiting for session 3434a6db-8a5d-4d2e-aad6-51d995e641e9 to get into ready status...
Session 3434a6db-8a5d-4d2e-aad6-51d995e641e9 ha

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog & Convert dynamicFrame to spark dataframe


In [3]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='bt-course-db-final', table_name='raw_data_2024_mm_06')
df = dyf.toDF()
df.show()

+---------+--------+-----------+--------------+---------------+--------------+---------------+--------------------+
|     date|    time|euipment id|equipment name| equipment type|attribute name|attribute value|                desc|
+---------+--------+-----------+--------------+---------------+--------------+---------------+--------------------+
|1/06/2024|17:30:28|   EQU-ID-1|  server-web-1|     web-server| CPUUtlization|             10|ideal should stay...|
|1/06/2024|17:30:28|   EQU-ID-2|  server-web-2|     web-server| CPUUtlization|             10|ideal should stay...|
|1/06/2024|17:30:28|   EQU-ID-3|  server-wev-3|     web-server| CPUUtlization|             10|ideal should stay...|
|1/06/2024|17:30:28|   EQU-ID-4|  server-app-1|     app-server| CPUUtlization|             10|ideal should stay...|
|1/06/2024|17:30:28|   EQU-ID-5|  server-app-2|     app-server| CPUUtlization|             10|ideal should stay...|
|1/06/2024|17:30:28|   EQU-ID-6|  server-app-3|     app-server| CPUUtliz

#### Example: How to Drops fields within a DynamicFrame

In [7]:
df = df.drop("equipment type", "time")
df.show()

+---------+-----------+--------------+--------------+---------------+--------------------+
|     date|euipment id|equipment name|attribute name|attribute value|                desc|
+---------+-----------+--------------+--------------+---------------+--------------------+
|1/06/2024|   EQU-ID-1|  server-web-1| CPUUtlization|             10|ideal should stay...|
|1/06/2024|   EQU-ID-2|  server-web-2| CPUUtlization|             10|ideal should stay...|
|1/06/2024|   EQU-ID-3|  server-wev-3| CPUUtlization|             10|ideal should stay...|
|1/06/2024|   EQU-ID-4|  server-app-1| CPUUtlization|             10|ideal should stay...|
|1/06/2024|   EQU-ID-5|  server-app-2| CPUUtlization|             10|ideal should stay...|
|1/06/2024|   EQU-ID-6|  server-app-3| CPUUtlization|             10|ideal should stay...|
|1/06/2024|   EQU-ID-7|  server-app-4| CPUUtlization|             10|ideal should stay...|
|1/06/2024|   EQU-ID-8|  server-app-5| CPUUtlization|             10|ideal should stay...|

#### Example: Drops all null fields in a DynamicFrame whose type is NullType

In [17]:
# Add new column "empty_column" with NullType
from pyspark.sql.functions import col,lit
from pyspark.sql.types import NullType
from awsglue.dynamicframe import DynamicFrame

df_with_nulls = dyf.toDF().withColumn("empty_column", lit(None).cast(NullType()))
df_with_nulls.show()
dyf_with_nulls= DynamicFrame.fromDF(df_with_nulls, glueContext, "df_with_nulls")
print("Schema for the dyf_with_nulls_dyf DynamicFrame:")
dyf_with_nulls.printSchema()

# Remove the NullType field
dyf_no_nulls = DropNullFields.apply(dyf_with_nulls)
print("Schema for the dyf_no_nulls DynamicFrame:")
dyf_no_nulls.printSchema()
df=dyf_no_nulls.toDF()
df.show()

+---------+--------+-----------+--------------+---------------+--------------+---------------+--------------------+------------+
|     date|    time|euipment id|equipment name| equipment type|attribute name|attribute value|                desc|empty_column|
+---------+--------+-----------+--------------+---------------+--------------+---------------+--------------------+------------+
|1/06/2024|17:30:28|   EQU-ID-1|  server-web-1|     web-server| CPUUtlization|             10|ideal should stay...|        null|
|1/06/2024|17:30:28|   EQU-ID-2|  server-web-2|     web-server| CPUUtlization|             10|ideal should stay...|        null|
|1/06/2024|17:30:28|   EQU-ID-3|  server-wev-3|     web-server| CPUUtlization|             10|ideal should stay...|        null|
|1/06/2024|17:30:28|   EQU-ID-4|  server-app-1|     app-server| CPUUtlization|             10|ideal should stay...|        null|
|1/06/2024|17:30:28|   EQU-ID-5|  server-app-2|     app-server| CPUUtlization|             10|ide

#### Example : How to filter records in a  dataframe

In [31]:
# Using equal condition
#df.filter(df["equipment type"] == "web-server").show(truncate=False)


# Not equals condition
#df.filter(df["equipment type"] != "web-server").show(truncate=False)

# Another Negation operator expression
#df.filter(~(df["equipment type"] == "web-server")).show(truncate=False)

# Filter using and operator with multiple conditions
#df.filter((df["equipment type"] == "web-server") & (df["attribute name"] == "CPUUtlization")).show(truncate=False)  

# Filter using OR operator with multiple conditions
#df.filter((df["equipment type"] == "web-server") | (df["equipment type"] == "app-server") ).show(truncate=False) 

# Filter IS IN List values
#li=["web-server","app-server","SEN"]
#df.filter(df["equipment type"].isin(li)).show()

# Using startswith database
#df.filter(df["equipment type"].startswith("database")).show()

#using endswith
#df.filter(df["equipment type"].endswith("N")).show()

#contains
#df.filter(df["equipment type"].contains("base")).show()

# like - SQL LIKE pattern
df.filter(df["equipment type"].like("%app%")).show()




+----------+--------+-----------+--------------+--------------+--------------+---------------+--------------------+
|      date|    time|euipment id|equipment name|equipment type|attribute name|attribute value|                desc|
+----------+--------+-----------+--------------+--------------+--------------+---------------+--------------------+
| 1/06/2024|17:30:28|   EQU-ID-4|  server-app-1|    app-server| CPUUtlization|             10|ideal should stay...|
| 1/06/2024|17:30:28|   EQU-ID-5|  server-app-2|    app-server| CPUUtlization|             10|ideal should stay...|
| 1/06/2024|17:30:28|   EQU-ID-6|  server-app-3|    app-server| CPUUtlization|             10|ideal should stay...|
| 1/06/2024|17:30:28|   EQU-ID-7|  server-app-4|    app-server| CPUUtlization|             10|ideal should stay...|
| 1/06/2024|17:30:28|   EQU-ID-8|  server-app-5|    app-server| CPUUtlization|             10|ideal should stay...|
| 1/06/2024|17:30:28|   EQU-ID-9|  server-app-6|    app-server| CPUUtliz

#### Example : Write dynamic frame Dataset to AWS S3 using dynamic_frame_from_options

In [46]:
euip_dyf = DynamicFrame.fromDF(df, glueContext)
s3output = glueContext.write_dynamic_frame_from_options(frame = euip_dyf,
        connection_type = "s3",
        connection_options = {"path": "s3://bt-course-bucket-4/output/"},
        format = "csv",
        format_options={
        "separator": ","
   })
#s3output.show()
